# Setup: montar Drive y clonar repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_PATH = '/content/drive/MyDrive/tfg-datos'

if not os.path.exists(REPO_PATH):
    # Primera vez: clona el repo en Drive
    !git clone https://github.com/monica793/tfg-datos.git {REPO_PATH}
else:
    # Ya existe: actualiza con los últimos cambios de VS Code
    !git -C {REPO_PATH} pull

print('Repo listo en', REPO_PATH)

In [ ]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/tfg-datos')
print('Path añadido')

# Instalar Sionna (solo si no está instalado)

In [ ]:
try:
    import sionna.phy
    print('Sionna ya instalado')
except ImportError:
    !pip install sionna
    print('Sionna instalado, reinicia el runtime si hay errores')

# Entrenar (o cargar checkpoint si ya existe)

In [ ]:
import os
os.chdir('/content/drive/MyDrive/tfg-datos')

from training.train_hybrid import train_ae_for_n, k_is_valid_for_5g, N_FIXED, RHO_DBS, K_CAND

# Entrena o carga checkpoint para cada SNR
trained_models = {}
for rho_db in RHO_DBS:
    valid_ks = [k for k in K_CAND if k < N_FIXED and k_is_valid_for_5g(k, N_FIXED)]
    trained_models[rho_db] = train_ae_for_n(n=N_FIXED, rho_db=rho_db, valid_ks=valid_ks)

# Evaluar y generar curvas Pfa/Pmd/Pie vs R

In [ ]:
from evaluation.plot_pfa_pmd_pie import run_curves_for_n

for rho_db, ae in trained_models.items():
    run_curves_for_n(n=N_FIXED, rho_db=rho_db, ae=ae)